In [1]:
# ============================================================================
# Kaggle-Optimized: Bottleneck Formation in Deep Networks (CIFAR-10 Edition)
# Version 6 - Full Code with CIFAR-10, Time Management, and 14 Plots
# ============================================================================

# %% Cell 1: Imports and Setup
import os
import sys
import pickle
import hashlib
import time
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, asdict
from itertools import product
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.linalg import svd as scipy_svd
import pandas as pd

print("✓ Imports successful! (CIFAR-10 Production Version)")

# %% Cell 2: Kaggle Environment Setup
def setup_kaggle_environment():
    """Setup paths and check Kaggle environment."""
    if os.path.exists('/kaggle/working'):
        BASE_DIR = Path('/kaggle/working')
    else:
        BASE_DIR = Path('.')
    
    RESULTS_DIR = BASE_DIR / 'results'
    RESULTS_DIR.mkdir(exist_ok=True)
    
    PLOTS_DIR = BASE_DIR / 'plots'
    PLOTS_DIR.mkdir(exist_ok=True)
    
    CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
    CHECKPOINT_DIR.mkdir(exist_ok=True)
    
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    print(f"✓ Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✓ Results directory: {RESULTS_DIR}")
    print(f"✓ Plots directory: {PLOTS_DIR}")
    
    return RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE

RESULTS_DIR, PLOTS_DIR, CHECKPOINT_DIR, DEVICE = setup_kaggle_environment()

# %% Cell 3: Time and Memory Management (Optimized)

class ExperimentTimer:
    """Manage experiment time to stay within Kaggle limits."""
    
    def __init__(self, max_hours: float = 11.5):
        self.start_time = time.time()
        self.max_seconds = max_hours * 3600
        
    def elapsed_hours(self) -> float:
        return (time.time() - self.start_time) / 3600
    
    def remaining_hours(self) -> float:
        return self.max_seconds / 3600 - self.elapsed_hours()
    
    def should_stop(self) -> bool:
        return self.elapsed_hours() >= (self.max_seconds / 3600)
    
    def print_status(self):
        print(f"⏱️  Elapsed: {self.elapsed_hours():.2f}h | Remaining: {self.remaining_hours():.2f}h")

def cleanup_memory():
    """Cleanup only when necessary (called per-run, not per-step)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_memory_stats():
    """Get current memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        cached = torch.cuda.memory_reserved() / 1e9
        return f"GPU: {allocated:.2f}GB allocated, {cached:.2f}GB cached"
    return "CPU mode"

# %% Cell 4: Reproducibility
def set_seed(seed: int):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    pl.seed_everything(seed, workers=True)



# %% Cell 5: Dataset Generation (CIFAR-10 with Global Caching)

# GLOBAL CACHE: Load CIFAR once, reuse across all runs
_CIFAR_CACHE = {'train': None, 'test': None}

def _initialize_cifar_cache(root: str = './data', device: str = 'cuda'):
    """Load full CIFAR-10 once to GPU memory."""
    global _CIFAR_CACHE
    
    if _CIFAR_CACHE['train'] is not None:
        return  # Already cached
    
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2023, 0.1994, 0.2010)
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    try:
        train_full = torchvision.datasets.CIFAR10(root=root, train=True, download=True, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=root, train=False, download=True, transform=transform)
    except:
        train_full = torchvision.datasets.CIFAR10(root=root, train=True, download=False, transform=transform)
        test_full = torchvision.datasets.CIFAR10(root=root, train=False, download=False, transform=transform)
    
    # Move to GPU
    train_loader_temp = DataLoader(train_full, batch_size=len(train_full), shuffle=False)
    test_loader_temp = DataLoader(test_full, batch_size=len(test_full), shuffle=False)
    
    X_train, y_train = next(iter(train_loader_temp))
    X_test, y_test = next(iter(test_loader_temp))
    
    _CIFAR_CACHE['train'] = {
        'X': X_train.view(len(X_train), -1).to(device),  # Flatten to 3072
        'y': y_train.to(device)
    }
    _CIFAR_CACHE['test'] = {
        'X': X_test.view(len(X_test), -1).to(device),
        'y': y_test.to(device)
    }
    
    print(f"✓ CIFAR-10 cached to {device}: Train={len(y_train)}, Test={len(y_test)}")


#== INITIALIZE CACHE IMMEDIATELY AFTER DEFINITION ===
print("\n" + "="*80)
print("📦 INITIALIZING GLOBAL CIFAR-10 CACHE")
print("="*80)
_initialize_cifar_cache(device=DEVICE)
print("="*80 + "\n")


class CIFAR10Subset(Dataset):
    """Subset selector from pre-cached global data."""
    def __init__(self, n_per_class: int, train: bool = True, seed: int = 42):
        global _CIFAR_CACHE
        self.n_per_class = n_per_class
        self.train = train
        
        cache_key = 'train' if train else 'test'
        X_full = _CIFAR_CACHE[cache_key]['X']
        y_full = _CIFAR_CACHE[cache_key]['y']
        
        if train and n_per_class > 0:
            # Select n_per_class from each of 10 classes
            selected_indices = []
            rng = np.random.RandomState(seed)
            
            for k in range(10):
                class_indices = torch.where(y_full == k)[0].cpu().numpy()
                rng.shuffle(class_indices)
                selected_indices.extend(class_indices[:min(n_per_class, len(class_indices))])
            
            self.X = X_full[selected_indices]
            self.y = y_full[selected_indices]
        else:
            self.X = X_full
            self.y = y_full

    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def create_data_loaders(K: int, d0: int, n_per_class: int, 
                        batch_size: int, seed: int = 42):
    """Create loaders from cached data."""
    train_full = CIFAR10Subset(n_per_class, train=True, seed=seed)
    
    N = len(train_full)
    n_val = max(10, int(0.1 * N))
    if n_val >= N: n_val = 1  # FIX: Always have 1 validation sample minimum
    n_train = N - n_val
    
    train_ds, val_ds = random_split(
        train_full, [n_train, n_val],
        generator=torch.Generator().manual_seed(seed)
    )
    
    test_ds = CIFAR10Subset(n_per_class=0, train=False)
    
    eff_batch_size = batch_size if batch_size > 0 else min(1024, len(train_ds))
    
    # No num_workers needed - data already on GPU
    train_loader = DataLoader(train_ds, batch_size=eff_batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=len(val_ds), shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=len(test_ds), shuffle=False, num_workers=0)
    
    return train_loader, val_loader, test_loader

# %% Cell 6: Network Architecture (CIFAR Adapted)
class BottleneckMLP(nn.Module):
    """
    Deep MLP for CIFAR-10.
    Input(3072) -> Project(d0) -> ReLU -> [Hidden Layers] -> Output(K)
    """
    
    def __init__(self, d0: int, K: int, L_hidden: int, d_b: int,
                 width_pattern: str = 'constant_narrow', use_relu: bool = True):
        super().__init__()
        self.d0 = d0 # 1000
        self.K = K
        
        # CIFAR Input Projection: 3072 -> d0
        self.input_dim = 3072
        self.projector = nn.Sequential(
            nn.Linear(self.input_dim, d0),
            nn.ReLU() if use_relu else nn.Identity()
        )
        
        self.L_hidden = L_hidden
        self.d_b = d_b
        self.width_pattern = width_pattern
        self.use_relu = use_relu
        
        widths = self._build_widths()
        self.widths = widths
        
        layers = []
        for i in range(len(widths) - 1):
            layers.append(nn.Linear(widths[i], widths[i+1]))
            if i < len(widths) - 2:
                if use_relu:
                    layers.append(nn.ReLU())
        
        self.network = nn.Sequential(*layers)
        
    def _build_widths(self) -> List[int]:
        if self.width_pattern == 'constant_narrow':
            widths = [self.d0] + [self.d_b] * self.L_hidden + [self.K]
        elif self.width_pattern == 'wide_narrow_wide':
            if self.L_hidden < 3:
                widths = [self.d0, self.d_b, self.K]
            else:
                n_wide = (self.L_hidden - 1) // 2
                widths = ([self.d0] + [512] * n_wide + [self.d_b] + 
                         [512] * (self.L_hidden - n_wide - 1) + [self.K])
        else:
            raise ValueError(f"Unknown width pattern: {self.width_pattern}")
        return widths
    
    def forward(self, x):
        h = self.projector(x)
        return self.network(h)
    
    def get_layer_activations(self, x: torch.Tensor) -> List[torch.Tensor]:
        """Extract activations including projection."""
        h = self.projector(x)
        activations = [h.detach()] # Layer 0 is after projection
        
        curr = h
        for layer in self.network:
            curr = layer(curr)
            if isinstance(layer, (nn.Linear, nn.ReLU)):
                activations.append(curr.detach())
        return activations

# %% Cell 7: Metrics Computation (OPTIMIZED)

class MetricsComputer:
    
    @staticmethod
    def stable_rank(H: torch.Tensor) -> float:
        """Stable rank: ||H||_F^2 / ||H||_2^2"""
        try:
            if H.shape[0] < 2: return float(H.shape[1])
            frob_norm = torch.norm(H, p='fro').item() ** 2
            spec_norm = torch.linalg.matrix_norm(H, ord=2).item() ** 2
            if not np.isfinite(frob_norm) or not np.isfinite(spec_norm) or spec_norm < 1e-10:
                return float(H.shape[1])
            return max(1.0, frob_norm / spec_norm)
        except: 
            return float(H.shape[1])

    @staticmethod
    def nc1_metric(H: torch.Tensor, y: torch.Tensor, K: int) -> float:
        """Optimized NC1 using PyTorch (stays on GPU)."""
        try:
            if not torch.isfinite(H).all(): return np.nan
            
            mu_G = H.mean(dim=0)
            Sigma_W = torch.zeros(H.shape[1], H.shape[1], device=H.device)
            Sigma_T = torch.zeros_like(Sigma_W)
            
            for k in range(K):
                mask = (y == k)
                H_k = H[mask]
                if len(H_k) == 0: continue
                
                mu_k = H_k.mean(dim=0)
                centered = H_k - mu_k
                Sigma_W += centered.T @ centered
                
                diff = (mu_k - mu_G).unsqueeze(1)
                Sigma_T += len(H_k) * (diff @ diff.T)
            
            Sigma_T += Sigma_W
            
            tr_W = torch.trace(Sigma_W).item()
            tr_T = torch.trace(Sigma_T).item()
            
            return float(np.clip(tr_W / (tr_T + 1e-10), 0.0, 1.0))
        except:
            return np.nan

    @staticmethod
    def singular_values(H: torch.Tensor, k: int = 10) -> np.ndarray:
        try:
            if H.shape[0] < k: k = H.shape[0]
            if not torch.isfinite(H).all(): return np.ones(k)
            U, S, V = torch.svd_lowrank(H, q=min(k, min(H.shape)))
            return S.cpu().numpy()
        except:
            return np.ones(k)

    @staticmethod
    def compute_all_metrics(model: nn.Module, X_train: torch.Tensor, y_train: torch.Tensor,
                           X_test: torch.Tensor, y_test: torch.Tensor, K: int) -> Dict:
        """
        OPTIMIZED: Takes pre-loaded tensors, no DataLoader overhead.
        Computes metrics + test accuracy in single pass.
        """
        model.eval()
        
        with torch.no_grad():
            # 1. Extract activations ONCE (efficient)
            activations = model.get_layer_activations(X_train)
            
            # 2. Layer metrics
            metrics = {}
            L = len(activations) - 1
            
            for ell in range(1, L + 1):
                H = activations[ell]
                if len(H.shape) == 1: H = H.unsqueeze(0)
                if ell == L or H.shape[1] == K: continue
                
                metrics[f'layer_{ell}'] = {
                    'stable_rank': MetricsComputer.stable_rank(H),
                    'nc1': MetricsComputer.nc1_metric(H, y_train, K),
                    'singular_values': MetricsComputer.singular_values(H, k=10),
                    'width': H.shape[1]
                }
            
            # 3. Test accuracy (single batch)
            logits_test = model(X_test)
            preds = logits_test.argmax(dim=1)
            test_acc = (preds == y_test).float().mean().item()
            metrics['test_acc_full'] = test_acc
        
        return metrics

# %% Cell 8: Experiment Configuration
@dataclass
class ExperimentConfig:
    K: int = 10
    d0: int = 1000
    n_per_class: int = 50
    d_b: int = 64
    L_hidden: int = 4
    width_pattern: str = 'constant_narrow'
    wd: float = 1e-3
    lr: float = 0.05
    max_iters: int = 100000 
    batch_size: int = -1
    seed: int = 42
    use_relu: bool = True
    
    def to_hash(self) -> str:
        config_dict = asdict(self)
        config_dict.pop('seed')
        config_str = str(sorted(config_dict.items()))
        return hashlib.md5(config_str.encode()).hexdigest()[:8]

# %% Cell 9: Lightning Module
class BottleneckLightningModule(pl.LightningModule):
    
    def __init__(self, config: ExperimentConfig):
        super().__init__()
        self.save_hyperparameters()
        self.config = config
        
        self.model = BottleneckMLP(
            d0=config.d0,
            K=config.K,
            L_hidden=config.L_hidden,
            d_b=config.d_b,
            width_pattern=config.width_pattern,
            use_relu=config.use_relu
        )
        
        self.criterion = nn.CrossEntropyLoss()
        
    def forward(self, x):
        return self.model(x)
    
    def training_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        X, y = batch
        logits = self(X)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', acc, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(),
            lr=self.config.lr,
            momentum=0.9,
            weight_decay=self.config.wd
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.config.max_iters
        )
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'interval': 'step'}}

# %% Cell 10: Training with Efficient Logging

def train_single_run(config: ExperimentConfig, timer: ExperimentTimer) -> Optional[Dict]:
    """Fast PyTorch loop with efficient logging (adaptive frequency)."""
    if timer.should_stop(): return None
    
    set_seed(config.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    try:
        # 1. Get data from cache
        train_full = CIFAR10Subset(config.n_per_class, train=True, seed=config.seed)
        
        N = len(train_full)
        n_val = max(1, int(0.1 * N))
        n_train = N - n_val
        
        train_ds, val_ds = random_split(
            train_full, [n_train, n_val],
            generator=torch.Generator().manual_seed(config.seed)
        )
        
        X_train = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
        y_train = torch.stack([train_ds[i][1] for i in range(len(train_ds))])
        
        X_val = torch.stack([val_ds[i][0] for i in range(len(val_ds))])
        y_val = torch.stack([val_ds[i][1] for i in range(len(val_ds))])
        
        X_test = _CIFAR_CACHE['test']['X']
        y_test = _CIFAR_CACHE['test']['y']
        
        # 2. Model
        model = BottleneckMLP(
            d0=config.d0, K=config.K, L_hidden=config.L_hidden, 
            d_b=config.d_b, width_pattern=config.width_pattern, 
            use_relu=config.use_relu
        ).to(device)
        
        optimizer = torch.optim.SGD(
            model.parameters(), lr=config.lr, momentum=0.9, weight_decay=config.wd
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=config.max_iters
        )
        criterion = nn.CrossEntropyLoss()
        
        # === EFFICIENT LOGGING: Adaptive frequency based on max_iters ===
        # Log ~50 points total (not too sparse, not too dense)
        log_interval = max(100, config.max_iters // 50)
        
        training_history = {
            'step': [],
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'test_acc': [],
            'learning_rate': [],
            'grad_norm': []  # Useful for debugging
        }
        
        # 3. Training loop
        best_val_loss = float('inf')
        patience = 5000
        patience_counter = 0
        
        model.train()
        
        for step in range(config.max_iters):
            optimizer.zero_grad(set_to_none=True)
            logits = model(X_train)
            loss = criterion(logits, y_train)
            loss.backward()
            
            # Gradient norm (cheap to compute, useful metric)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
            
            optimizer.step()
            scheduler.step()
            
            # === EFFICIENT LOGGING: Only at intervals ===
            if step % log_interval == 0 or step == config.max_iters - 1:
                model.eval()
                with torch.no_grad():
                    # Train metrics (already computed)
                    train_loss = loss.item()
                    train_acc = (logits.argmax(dim=1) == y_train).float().mean().item()
                    
                    # Val metrics
                    val_logits = model(X_val)
                    val_loss = criterion(val_logits, y_val).item()
                    val_acc = (val_logits.argmax(dim=1) == y_val).float().mean().item()
                    
                    # Test metrics (single forward pass - efficient)
                    test_logits = model(X_test)
                    test_acc = (test_logits.argmax(dim=1) == y_test).float().mean().item()
                    
                    # Store history
                    training_history['step'].append(step)
                    training_history['train_loss'].append(train_loss)
                    training_history['train_acc'].append(train_acc)
                    training_history['val_loss'].append(val_loss)
                    training_history['val_acc'].append(val_acc)
                    training_history['test_acc'].append(test_acc)
                    training_history['learning_rate'].append(optimizer.param_groups[0]['lr'])
                    training_history['grad_norm'].append(grad_norm.item())
                
                model.train()
            
            # Early stopping check (every 1000 steps)
            if step % 1000 == 0 and step > 0:
                model.eval()
                with torch.no_grad():
                    val_logits = model(X_val)
                    val_loss = criterion(val_logits, y_val).item()
                model.train()
                
                if val_loss < best_val_loss - 1e-4:
                    best_val_loss = val_loss
                    patience_counter = 0
                else:
                    patience_counter += 1000
                    
                if patience_counter >= patience:
                    break
        
        # 4. Final metrics
        model.eval()
        metrics = MetricsComputer.compute_all_metrics(
            model, X_train, y_train, X_test, y_test, config.K
        )
        
        # Final accuracies
        with torch.no_grad():
            final_train_acc = (model(X_train).argmax(dim=1) == y_train).float().mean().item()
            final_val_acc = (model(X_val).argmax(dim=1) == y_val).float().mean().item()
            final_test_acc = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
        
        result = {
            'config': asdict(config),
            'metrics': metrics,
            'training_history': training_history,
            'final_train_acc': final_train_acc,
            'final_val_acc': final_val_acc,
            'final_test_acc': final_test_acc,
            'final_val_loss': best_val_loss,
            'steps_taken': step + 1,
            'converged': (patience_counter >= patience)
        }
        
        # Cleanup
        del model, optimizer
        cleanup_memory()
        
        return result
        
    except Exception as e:
        print(f" Run failed: {e}")
        cleanup_memory()
        return None


# %% Cell 11: Optimized Checkpoint Management

import threading
_checkpoint_lock = threading.Lock()

def save_checkpoint_async(results: List[Dict], checkpoint_path: Path):
    """Non-blocking checkpoint save."""
    def _save():
        with _checkpoint_lock:
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)
    
    thread = threading.Thread(target=_save, daemon=True)
    thread.start()

def save_checkpoint(results: List[Dict], checkpoint_path: Path):
    """Synchronous save for final checkpoint."""
    with open(checkpoint_path, 'wb') as f:
        pickle.dump(results, f)
    print(f"💾 Checkpoint saved: {len(results)} results")

def load_checkpoint(checkpoint_path: Path) -> List[Dict]:
    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            results = pickle.load(f)
        print(f"Loaded checkpoint: {len(results)} results")
        return results
    return []

# %% Cell 12: Sweep with Automatic Fallback Plotting

def run_sweep(sweep_params: Dict, n_seeds: int, timer: ExperimentTimer, 
              checkpoint_path: Path, plots_dir: Path, results_dir: Path) -> List[Dict]:
    """Sweep with automatic plotting fallback when time runs out."""
    
    all_results = load_checkpoint(checkpoint_path)
    
    keys = list(sweep_params.keys())
    values = [sweep_params[k] for k in keys]
    
    completed_configs = set()
    for res in all_results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'], 
               cfg['width_pattern'], cfg['wd'], cfg['seed'])
        completed_configs.add(key)
    
    total_configs = np.prod([len(v) for v in values]) * n_seeds
    config_count = 0
    
    print(f"\n{'='*80}")
    print(f"RESUMING SWEEP: {len(all_results)}/{total_configs} runs already completed")
    print(f"{'='*80}\n")
    
    try:
        for combo in product(*values):
            config_dict = dict(zip(keys, combo))
            
            for seed in range(n_seeds):
                config = ExperimentConfig(**config_dict, seed=seed)
                
                key = (config.n_per_class, config.d_b, config.L_hidden,
                       config.width_pattern, config.wd, seed)
                if key in completed_configs:
                    continue
                
                config_count += 1
                
                # === TIME CHECK: 45min buffer for plotting ===
                if timer.remaining_hours() < 0.75:  # 45 minutes left
                    print(f"\n{'='*80}")
                    print(f" TIME WARNING: Only {timer.remaining_hours():.2f}h remaining")
                    print(f" Saving checkpoint and generating plots...")
                    print(f"{'='*80}\n")
                    
                    # Save final checkpoint
                    save_checkpoint(all_results, checkpoint_path)
                    
                    # Generate plots with current data
                    generate_all_plots(all_results, plots_dir, results_dir)
                    
                    print(f"\n{'='*80}")
                    print(f" GRACEFUL SHUTDOWN COMPLETE")
                    print(f" {len(all_results)} runs completed, plots saved")
                    print(f" Resume by re-running this notebook (will auto-continue)")
                    print(f"{'='*80}\n")
                    
                    return all_results
                
                print(f"\n[{len(all_results)+1}/{total_configs}] n={config.n_per_class}, "
                      f"d_b={config.d_b}, L={config.L_hidden}, wd={config.wd}, seed={seed}")
                timer.print_status()
                
                result = train_single_run(config, timer)
                if result is not None:
                    all_results.append(result)
                    
                    # Save every 20 runs
                    if len(all_results) % 20 == 0:
                        save_checkpoint_async(all_results.copy(), checkpoint_path)
                        print(f"  Checkpoint saved ({len(all_results)} runs)")
            
            # Check time between configs too
            if timer.remaining_hours() < 0.75:
                break
        
        # Normal completion
        save_checkpoint(all_results, checkpoint_path)
        print(f"\n SWEEP COMPLETED! All {len(all_results)} runs finished.")
        
    except KeyboardInterrupt:
        print(f"\n  Manual interruption detected!")
        save_checkpoint(all_results, checkpoint_path)
        generate_all_plots(all_results, plots_dir, results_dir)
        print(f" Progress saved, plots generated")
        
    except Exception as e:
        print(f"\n Error during sweep: {e}")
        save_checkpoint(all_results, checkpoint_path)
        print(f" Emergency checkpoint saved")
        raise
    
    return all_results

# %% Cell 13: VISUALIZATION FUNCTIONS (Full Set of 14 Plots)

def plot_rank_evolution(results: List[Dict], save_path: Path):
    """Stable rank across layers for different configs."""
    if len(results) == 0: return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['width_pattern'])
        if key not in config_groups: config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, pattern = key
        layer_ranks = {}
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics['stable_rank']
                    if np.isfinite(rank):
                        if layer_idx not in layer_ranks: layer_ranks[layer_idx] = []
                        layer_ranks[layer_idx].append(rank)
        
        if not layer_ranks: continue
        layers = sorted(layer_ranks.keys())
        means = [np.mean(layer_ranks[l]) for l in layers]
        stds = [np.std(layer_ranks[l]) for l in layers]
        ax.plot(layers, means, 'o-', linewidth=2, label='Stable Rank')
        ax.fill_between(layers, np.array(means)-np.array(stds), np.array(means)+np.array(stds), alpha=0.3)
        ax.set_title(f'Rank Evolution: n={n}, d_b={d_b}')
        ax.set_xlabel('Layer Index')
        ax.set_ylabel('Stable Rank')
        ax.grid(True, alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_singular_spectra(results: List[Dict], save_path: Path):
    if len(results) == 0: return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups: config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax = axes[idx]
        n, d_b, L = key
        for res in group[:1]:
            metrics = res['metrics']
            middle_layer = L // 2 + 1
            layer_key = f'layer_{middle_layer}'
            if layer_key in metrics:
                svals = metrics[layer_key]['singular_values']
                if len(svals) > 0: ax.loglog(range(1, len(svals)+1), svals, 'o-')
        ax.set_title(f'n={n}, d_b={d_b}, L={L}')
        ax.grid(True)
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_bottleneck_location(results: List[Dict], save_path: Path):
    bottleneck_layers = []
    for res in results:
        metrics = res['metrics']
        ranks = {}
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): ranks[int(layer_name.split('_')[1])] = rank
        if len(ranks) > 1: bottleneck_layers.append(min(ranks, key=ranks.get))
    if not bottleneck_layers: return
    plt.figure(figsize=(10, 6))
    plt.hist(bottleneck_layers, bins=max(10, len(set(bottleneck_layers))), edgecolor='black')
    plt.title('Distribution of Bottleneck Layer Location')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_scaling_law(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'nK': cfg['n_per_class']*cfg['K'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(8, 6))
    plt.loglog(df['nK'], df['r_min'], 'o', alpha=0.5)
    plt.xlabel('nK')
    plt.ylabel('Min Stable Rank')
    plt.title('Scaling Law')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_phase_diagram(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            nK = cfg['n_per_class'] * cfg['K']
            data.append({
                'log_n': np.log10(cfg['n_per_class']),
                'log_d_b': np.log10(cfg['d_b']),
                'norm_rank': min_rank / min(cfg['d_b'], nK)
            })
    if not data: return
    df = pd.DataFrame(data)
    pivot = df.pivot_table(values='norm_rank', index='log_d_b', columns='log_n', aggfunc='mean')
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, cmap='viridis')
    plt.title('Phase Diagram')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_training_dynamics(results: List[Dict], save_path: Path):
    """NC1 vs Stable Rank dual-axis plot across layers."""
    if len(results) == 0: return
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    config_groups = {}
    for res in results:
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups: config_groups[key] = []
        config_groups[key].append(res)
    
    for idx, (key, group) in enumerate(list(config_groups.items())[:6]):
        ax1 = axes[idx]
        n, d_b, L = key
        
        layer_ranks = {}
        layer_nc1 = {}
        
        for res in group:
            metrics = res['metrics']
            for layer_name, layer_metrics in metrics.items():
                if layer_name.startswith('layer_'):
                    layer_idx = int(layer_name.split('_')[1])
                    rank = layer_metrics['stable_rank']
                    nc1 = layer_metrics['nc1']
                    if np.isfinite(rank):
                        if layer_idx not in layer_ranks: layer_ranks[layer_idx] = []
                        layer_ranks[layer_idx].append(rank)
                    if np.isfinite(nc1):
                        if layer_idx not in layer_nc1: layer_nc1[layer_idx] = []
                        layer_nc1[layer_idx].append(nc1)
        
        if not layer_ranks: continue
        
        layers = sorted(layer_ranks.keys())
        rank_means = [np.mean(layer_ranks[l]) for l in layers]
        
        # Plot Stable Rank
        color1 = 'tab:blue'
        ax1.set_xlabel('Layer Index')
        ax1.set_ylabel('Stable Rank', color=color1)
        ax1.plot(layers, rank_means, 'o-', color=color1, linewidth=2, label='Stable Rank')
        ax1.tick_params(axis='y', labelcolor=color1)
        ax1.grid(True, alpha=0.3)
        
        # Dual axis for NC1
        if layer_nc1:
            ax2 = ax1.twinx()
            nc1_layers = sorted(layer_nc1.keys())
            nc1_means = [np.mean(layer_nc1[l]) for l in nc1_layers]
            color2 = 'tab:red'
            ax2.set_ylabel('NC1 (Within-Class Variance)', color=color2)
            ax2.plot(nc1_layers, nc1_means, 's--', color=color2, linewidth=2, label='NC1')
            ax2.tick_params(axis='y', labelcolor=color2)
        
        ax1.set_title(f'Training Dynamics: n={n}, d_b={d_b}, L={L}')
        
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_wd_ablation(results: List[Dict], save_path: Path):
    wd_values = sorted(list(set([r['config']['wd'] for r in results])))
    if not wd_values: return
    fig, axes = plt.subplots(1, min(len(wd_values), 4), figsize=(20, 5))
    if len(wd_values) == 1: axes = [axes]
    for idx, wd in enumerate(wd_values[:4]):
        subset = [r for r in results if r['config']['wd'] == wd]
        ranks = []
        for res in subset:
            metrics = res['metrics']
            curr_ranks = [m['stable_rank'] for k, m in metrics.items() if k.startswith('layer_')]
            if curr_ranks: ranks.append(np.mean(curr_ranks))
        if ranks:
            axes[idx].hist(ranks, bins=10)
            axes[idx].set_title(f'WD={wd}')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_depth_analysis(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'L': cfg['L_hidden'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=df, x='L', y='r_min', marker='o')
    plt.title('Depth Analysis')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_capacity_analysis(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'capacity': min(cfg['d_b'], cfg['n_per_class']*cfg['K']/2), 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(8, 8))
    plt.scatter(df['capacity'], df['r_min'])
    plt.plot([0, df['capacity'].max()], [0, df['capacity'].max()], 'r--')
    plt.title('Capacity Analysis')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_accuracy_vs_rank(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'r_min': min_rank, 'acc': res.get('final_train_acc', 0)})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    plt.scatter(df['r_min'], df['acc'])
    plt.title('Accuracy vs Rank')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_architecture_comparison(results: List[Dict], save_path: Path):
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'pattern': cfg['width_pattern'], 'r_min': min_rank})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=df, x='pattern', y='r_min')
    plt.title('Architecture Comparison')
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_test_acc_vs_rank(results: List[Dict], save_path: Path):
    """New: Test Accuracy vs Bottleneck Rank (Generalization)."""
    if len(results) < 5: return
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        test_acc = metrics.get('test_acc_full', 0)
        min_rank = float('inf')
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics['stable_rank']
                if np.isfinite(rank): min_rank = min(min_rank, rank)
        if min_rank < float('inf'):
            data.append({'r_min': min_rank, 'test_acc': test_acc, 'n': cfg['n_per_class']})
    if not data: return
    df = pd.DataFrame(data)
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x='r_min', y='test_acc', hue='n', palette='viridis', s=100)
    plt.title('Generalization vs Compression')
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_transfer_comparison(results: List[Dict], save_path: Path):
    """Compare rank evolution between different width patterns (transfer-like behavior)."""
    if len(results) == 0: return
    
    pattern_groups = {}
    for res in results:
        cfg = res['config']
        pattern = cfg['width_pattern']
        if pattern not in pattern_groups: pattern_groups[pattern] = []
        pattern_groups[pattern].append(res)
    
    fig, axes = plt.subplots(1, len(pattern_groups), figsize=(12 * len(pattern_groups), 8))
    if len(pattern_groups) == 1: axes = [axes]
    
    for idx, (pattern, group) in enumerate(pattern_groups.items()):
        ax = axes[idx]
        
        # Group by (n, d_b) within pattern
        config_subgroups = {}
        for res in group:
            cfg = res['config']
            key = (cfg['n_per_class'], cfg['d_b'])
            if key not in config_subgroups: config_subgroups[key] = []
            config_subgroups[key].append(res)
        
        for (n, d_b), subgroup in list(config_subgroups.items())[:5]:
            layer_ranks = {}
            for res in subgroup:
                metrics = res['metrics']
                for layer_name, layer_metrics in metrics.items():
                    if layer_name.startswith('layer_'):
                        layer_idx = int(layer_name.split('_')[1])
                        rank = layer_metrics['stable_rank']
                        if np.isfinite(rank):
                            if layer_idx not in layer_ranks: layer_ranks[layer_idx] = []
                            layer_ranks[layer_idx].append(rank)
            
            if not layer_ranks: continue
            layers = sorted(layer_ranks.keys())
            means = [np.mean(layer_ranks[l]) for l in layers]
            ax.plot(layers, means, 'o-', linewidth=2, label=f'n={n}, d_b={d_b}', alpha=0.7)
        
        ax.set_title(f'Transfer Analysis: {pattern}')
        ax.set_xlabel('Layer Index')
        ax.set_ylabel('Stable Rank')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

# %% Cell 13: Add Training Curves Plots

def plot_training_curves_grid(results: List[Dict], save_path: Path):
    """Grid of training curves for representative configurations."""
    if len(results) == 0: return
    
    # Select 6 representative configs (different n, d_b)
    config_groups = {}
    for res in results:
        if 'training_history' not in res or len(res['training_history']['step']) == 0:
            continue
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups: 
            config_groups[key] = res  # Take first seed
    
    selected = list(config_groups.values())[:6]
    if len(selected) == 0: return
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    
    for idx, res in enumerate(selected):
        ax = axes[idx]
        hist = res['training_history']
        cfg = res['config']
        
        steps = hist['step']
        
        # Plot train/val/test accuracy
        ax.plot(steps, hist['train_acc'], 'b-', linewidth=2, label='Train Acc', alpha=0.8)
        ax.plot(steps, hist['val_acc'], 'g--', linewidth=2, label='Val Acc', alpha=0.8)
        ax.plot(steps, hist['test_acc'], 'r:', linewidth=2.5, label='Test Acc', alpha=0.9)
        
        ax.set_title(f"n={cfg['n_per_class']}, d_b={cfg['d_b']}, L={cfg['L_hidden']}", fontsize=12)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Accuracy')
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1.05])
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_loss_curves_grid(results: List[Dict], save_path: Path):
    """Grid of loss curves (train vs val)."""
    if len(results) == 0: return
    
    config_groups = {}
    for res in results:
        if 'training_history' not in res or len(res['training_history']['step']) == 0:
            continue
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'], cfg['L_hidden'])
        if key not in config_groups: 
            config_groups[key] = res
    
    selected = list(config_groups.values())[:6]
    if len(selected) == 0: return
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    
    for idx, res in enumerate(selected):
        ax = axes[idx]
        hist = res['training_history']
        cfg = res['config']
        
        steps = hist['step']
        
        ax.semilogy(steps, hist['train_loss'], 'b-', linewidth=2, label='Train Loss', alpha=0.8)
        ax.semilogy(steps, hist['val_loss'], 'r--', linewidth=2, label='Val Loss', alpha=0.8)
        
        ax.set_title(f"n={cfg['n_per_class']}, d_b={cfg['d_b']}, L={cfg['L_hidden']}", fontsize=12)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Loss (log scale)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_generalization_gap(results: List[Dict], save_path: Path):
    """Train-Test accuracy gap across configurations."""
    data = []
    for res in results:
        if 'final_train_acc' not in res or 'final_test_acc' not in res:
            continue
        cfg = res['config']
        gap = res['final_train_acc'] - res['final_test_acc']
        data.append({
            'n': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L': cfg['L_hidden'],
            'train_acc': res['final_train_acc'],
            'test_acc': res['final_test_acc'],
            'gap': gap,
            'wd': cfg['wd']
        })
    
    if not data: return
    df = pd.DataFrame(data)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Left: Gap vs n (grouped by d_b)
    ax = axes[0]
    for d_b in sorted(df['d_b'].unique()):
        subset = df[df['d_b'] == d_b]
        grouped = subset.groupby('n')['gap'].mean()
        ax.plot(grouped.index, grouped.values, 'o-', linewidth=2, label=f'd_b={d_b}', markersize=8)
    
    ax.set_xlabel('n (samples per class)', fontsize=12)
    ax.set_ylabel('Train-Test Accuracy Gap', fontsize=12)
    ax.set_title('Generalization Gap vs Data Size', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    
    # Right: Scatter test vs train
    ax = axes[1]
    scatter = ax.scatter(df['train_acc'], df['test_acc'], c=df['n'], 
                        s=100, alpha=0.6, cmap='viridis', edgecolors='black', linewidth=0.5)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect Generalization')
    
    ax.set_xlabel('Train Accuracy', fontsize=12)
    ax.set_ylabel('Test Accuracy', fontsize=12)
    ax.set_title('Test vs Train Accuracy', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('n (samples/class)', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_learning_rate_schedule(results: List[Dict], save_path: Path):
    """Visualize learning rate decay during training."""
    if len(results) == 0: return
    
    # Pick one representative run
    for res in results:
        if 'training_history' in res and len(res['training_history']['step']) > 0:
            hist = res['training_history']
            cfg = res['config']
            
            fig, ax = plt.subplots(1, 1, figsize=(10, 6))
            
            ax.plot(hist['step'], hist['learning_rate'], 'b-', linewidth=2)
            ax.set_xlabel('Training Step', fontsize=12)
            ax.set_ylabel('Learning Rate', fontsize=12)
            ax.set_title(f'Cosine LR Schedule (max_iters={cfg["max_iters"]})', fontsize=14)
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f"✓ Saved: {save_path.name}")
            break

def plot_gradient_norms(results: List[Dict], save_path: Path):
    """Gradient norm evolution (helps detect training instability)."""
    if len(results) == 0: return
    
    config_groups = {}
    for res in results:
        if 'training_history' not in res or len(res['training_history']['step']) == 0:
            continue
        cfg = res['config']
        key = (cfg['n_per_class'], cfg['d_b'])
        if key not in config_groups: 
            config_groups[key] = res
    
    selected = list(config_groups.values())[:4]
    if len(selected) == 0: return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, res in enumerate(selected):
        ax = axes[idx]
        hist = res['training_history']
        cfg = res['config']
        
        ax.semilogy(hist['step'], hist['grad_norm'], 'purple', linewidth=1.5, alpha=0.7)
        ax.set_title(f"Gradient Norms: n={cfg['n_per_class']}, d_b={cfg['d_b']}", fontsize=12)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('Gradient Norm (log scale)')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def plot_convergence_speed(results: List[Dict], save_path: Path):
    """Steps to convergence vs configuration parameters."""
    data = []
    for res in results:
        if 'steps_taken' not in res: continue
        cfg = res['config']
        data.append({
            'n': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L': cfg['L_hidden'],
            'steps': res['steps_taken'],
            'converged': res.get('converged', False)
        })
    
    if not data: return
    df = pd.DataFrame(data)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Left: Steps vs n
    ax = axes[0]
    grouped = df.groupby('n')['steps'].agg(['mean', 'std'])
    ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], 
                fmt='o-', linewidth=2, markersize=8, capsize=5)
    ax.set_xlabel('n (samples per class)', fontsize=12)
    ax.set_ylabel('Steps to Convergence', fontsize=12)
    ax.set_title('Convergence Speed vs Data Size', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    # Right: Steps vs depth
    ax = axes[1]
    grouped = df.groupby('L')['steps'].agg(['mean', 'std'])
    ax.errorbar(grouped.index, grouped['mean'], yerr=grouped['std'], 
                fmt='s-', linewidth=2, markersize=8, capsize=5, color='green')
    ax.set_xlabel('Network Depth (L)', fontsize=12)
    ax.set_ylabel('Steps to Convergence', fontsize=12)
    ax.set_title('Convergence Speed vs Depth', fontsize=14)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: {save_path.name}")

def generate_summary_table(results: List[Dict], save_path: Path):
    if len(results) == 0: return
    data = []
    for res in results:
        cfg = res['config']
        metrics = res['metrics']
        ranks = [m['stable_rank'] for k, m in metrics.items() if k.startswith('layer_') and np.isfinite(m['stable_rank'])]
        if ranks:
            data.append({
                'n': cfg['n_per_class'], 'd_b': cfg['d_b'], 'L': cfg['L_hidden'],
                'r_min': min(ranks), 'test_acc': metrics.get('test_acc_full', 0)
            })
    if data:
        pd.DataFrame(data).to_csv(save_path)
        print(f"✓ Saved: {save_path.name}")

# %% Cell 13b: Centralized Plot Generation Function

def generate_all_plots(results: List[Dict], plots_dir: Path, results_dir: Path):
    """Generate all plots and exports. Safe to call anytime."""
    if len(results) == 0:
        print("  No results to plot")
        return
    
    print(f"\n{'='*80}")
    print(f" GENERATING VISUALIZATIONS FROM {len(results)} RUNS")
    print(f"{'='*80}\n")
    
    try:
        # Original plots (1-13)
        plot_rank_evolution(results, plots_dir / '01_rank_evolution.png')
        plot_singular_spectra(results, plots_dir / '02_singular_spectra.png')
        plot_bottleneck_location(results, plots_dir / '03_bottleneck_location.png')
        plot_scaling_law(results, plots_dir / '04_scaling_law.png')
        plot_phase_diagram(results, plots_dir / '05_phase_diagram.png')
        plot_training_dynamics(results, plots_dir / '06_training_dynamics.png')
        plot_wd_ablation(results, plots_dir / '07_wd_ablation.png')
        plot_depth_analysis(results, plots_dir / '08_depth_analysis.png')
        plot_capacity_analysis(results, plots_dir / '09_capacity_analysis.png')
        plot_accuracy_vs_rank(results, plots_dir / '10_accuracy_vs_rank.png')
        plot_architecture_comparison(results, plots_dir / '11_architecture_comparison.png')
        plot_test_acc_vs_rank(results, plots_dir / '12_test_acc_vs_rank.png')
        plot_transfer_comparison(results, plots_dir / '13_transfer_comparison.png')
        
        # Training plots (14-19)
        plot_training_curves_grid(results, plots_dir / '14_training_curves.png')
        plot_loss_curves_grid(results, plots_dir / '15_loss_curves.png')
        plot_generalization_gap(results, plots_dir / '16_generalization_gap.png')
        plot_learning_rate_schedule(results, plots_dir / '17_lr_schedule.png')
        plot_gradient_norms(results, plots_dir / '18_gradient_norms.png')
        plot_convergence_speed(results, plots_dir / '19_convergence_speed.png')
        
        # Export data
        export_comprehensive_results(results, results_dir)
        
        # Save raw results
        with open(results_dir / 'all_results.pkl', 'wb') as f:
            pickle.dump(results, f)
        
        print(f"\n All 19 plots generated successfully!")
        print(f" Results exported to {results_dir}")
        
    except Exception as e:
        print(f"  Plot generation partially failed: {e}")
        print(f" Results still saved to checkpoint")


# %% Cell 14: Quick Test Run
print("\n" + "="*80)
print(" RUNNING QUICK TEST (should complete in <1 minute)")
print("="*80)

test_timer = ExperimentTimer(max_hours=0.1)
test_config = ExperimentConfig(
    n_per_class=20, # Small subset
    d_b=16,
    L_hidden=3,
    max_iters=100,
    width_pattern='constant_narrow',
    wd=1e-3,
    seed=42
)

print(f"Testing with: n=20, d_b=16, L=3, max_iters=100")
test_result = train_single_run(test_config, test_timer)

if test_result is not None:
    print(" TEST PASSED! Proceeding to full sweep...")
    cleanup_memory()
else:
    print(" TEST FAILED! Check configuration and try again.")
    sys.exit(1)

# %% Cell 15: Optimized 10-Hour Sweep
print("\n" + "="*80)
print(" SMART SWEEP CONFIGURATION (Fits in 11.5h)")
print("="*80)

sweep_params = {
    'n_per_class': [5, 10, 20, 40, 80, 160, 320, 640],  # 8 values
    'd_b': [8, 16, 32, 64, 128, 256, 512],              # 7 values
    'L_hidden': [3, 6, 10, 15, 22],                     # 5 values
    'width_pattern': ['constant_narrow', 'wide_narrow_wide'],
    'wd': [0, 1e-5, 1e-4, 1e-3, 1e-2],                  # 5 values
    'K': [10]
}
n_seeds = 3

# Total: 8×7×5×2×5 = 2,800 configs × 4 seeds = 11,200 runs
# Time: ~9-10h (pushing limits)

# Total: 7×6×3×2×5 = 1,260 configs × 3 seeds = 3,780 runs
# Expected time: ~4.5-5h (faster per-run compensates for more runs)

n_seeds = 3 
max_iters = 100000 

total_configs = 1
for param_name, param_values in sweep_params.items():
    total_configs *= len(param_values)

total_runs = total_configs * n_seeds
print(f"Total runs planned: {total_runs}")
print("Expected completion: ~9.5 - 11 hours")
print("="*80)

ExperimentConfig.__dataclass_fields__['max_iters'].default = max_iters

# %% Cell 16: Run Full Sweep with Auto-Recovery

print("\n Starting/Resuming sweep...")
main_timer = ExperimentTimer(max_hours=11.5)
checkpoint_path = CHECKPOINT_DIR / 'sweep_checkpoint.pkl'

all_results = run_sweep(
    sweep_params=sweep_params,
    n_seeds=n_seeds,
    timer=main_timer,
    checkpoint_path=checkpoint_path,
    plots_dir=PLOTS_DIR,      # NEW: Pass plotting paths
    results_dir=RESULTS_DIR   # NEW: Pass results paths
)

print(f"\n Session complete! {len(all_results)} results collected.")
print(f"  Total time: {main_timer.elapsed_hours():.2f} hours")

# Check if sweep is complete
total_expected = np.prod([len(v) for v in sweep_params.values()]) * n_seeds
if len(all_results) >= total_expected:
    print(f" FULL SWEEP COMPLETED!")
else:
    remaining = total_expected - len(all_results)
    print(f" {remaining} runs remaining - re-run notebook to continue")

# %% Cell 17: Generate Comprehensive Summary Statistics

def export_comprehensive_results(all_results: List[Dict], save_dir: Path):
    """Export detailed CSV with all metrics."""
    if len(all_results) == 0: return
    
    rows = []
    for res in all_results:
        cfg = res['config']
        metrics = res.get('metrics', {})
        
        # Basic config
        row = {
            'n_per_class': cfg['n_per_class'],
            'd_b': cfg['d_b'],
            'L_hidden': cfg['L_hidden'],
            'width_pattern': cfg['width_pattern'],
            'wd': cfg['wd'],
            'lr': cfg['lr'],
            'seed': cfg['seed'],
        }
        
        # Final accuracies
        row['final_train_acc'] = res.get('final_train_acc', np.nan)
        row['final_val_acc'] = res.get('final_val_acc', np.nan)
        row['final_test_acc'] = res.get('final_test_acc', np.nan)
        row['final_val_loss'] = res.get('final_val_loss', np.nan)
        
        # Generalization gap
        row['train_test_gap'] = row['final_train_acc'] - row['final_test_acc']
        
        # Training dynamics
        row['steps_taken'] = res.get('steps_taken', np.nan)
        row['converged'] = res.get('converged', False)
        
        # Bottleneck metrics
        min_rank = float('inf')
        min_nc1 = float('inf')
        bottleneck_layer = -1
        
        for layer_name, layer_metrics in metrics.items():
            if layer_name.startswith('layer_'):
                rank = layer_metrics.get('stable_rank', np.nan)
                nc1 = layer_metrics.get('nc1', np.nan)
                if np.isfinite(rank) and rank < min_rank:
                    min_rank = rank
                    bottleneck_layer = int(layer_name.split('_')[1])
                if np.isfinite(nc1):
                    min_nc1 = min(min_nc1, nc1)
        
        row['min_stable_rank'] = min_rank if min_rank < float('inf') else np.nan
        row['min_nc1'] = min_nc1 if min_nc1 < float('inf') else np.nan
        row['bottleneck_layer'] = bottleneck_layer
        
        # Normalized rank
        nK = cfg['n_per_class'] * cfg['K']
        row['normalized_rank'] = row['min_stable_rank'] / min(cfg['d_b'], nK) if np.isfinite(row['min_stable_rank']) else np.nan
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    
    # Main results
    df.to_csv(save_dir / 'detailed_results.csv', index=False)
    
    # Aggregated stats (mean over seeds)
    agg_df = df.groupby(['n_per_class', 'd_b', 'L_hidden', 'width_pattern', 'wd']).agg({
        'final_train_acc': ['mean', 'std'],
        'final_test_acc': ['mean', 'std'],
        'train_test_gap': ['mean', 'std'],
        'min_stable_rank': ['mean', 'std'],
        'min_nc1': ['mean', 'std'],
        'steps_taken': ['mean', 'std']
    }).reset_index()
    
    agg_df.columns = ['_'.join(col).strip('_') for col in agg_df.columns.values]
    agg_df.to_csv(save_dir / 'aggregated_results.csv', index=False)
    
    print(f"✓ Saved detailed_results.csv ({len(df)} runs)")
    print(f"✓ Saved aggregated_results.csv ({len(agg_df)} configs)")
    
    # Print summary statistics
    print("\n" + "="*60)
    print("FINAL SUMMARY STATISTICS")
    print("="*60)
    print(f"Total runs completed: {len(df)}")
    print(f"Unique configurations: {len(agg_df)}")
    print(f"\nTest Accuracy:")
    print(f"  Mean: {df['final_test_acc'].mean():.4f} ± {df['final_test_acc'].std():.4f}")
    print(f"  Max:  {df['final_test_acc'].max():.4f}")
    print(f"  Min:  {df['final_test_acc'].min():.4f}")
    print(f"\nGeneralization Gap:")
    print(f"  Mean: {df['train_test_gap'].mean():.4f} ± {df['train_test_gap'].std():.4f}")
    print(f"\nBottleneck Rank:")
    print(f"  Mean: {df['min_stable_rank'].mean():.4f} ± {df['min_stable_rank'].std():.4f}")
    print(f"  Min:  {df['min_stable_rank'].min():.4f}")
    print("="*60)

# %% Cell 17: Generate All Plots
print("\n" + "="*80)
print(" GENERATING VISUALIZATIONS")
print("="*80)

if len(all_results) > 0:
    print(f"Processing {len(all_results)} results...")
    
    # === ORIGINAL PLOTS ===
    plot_rank_evolution(all_results, PLOTS_DIR / '01_rank_evolution.png')
    plot_singular_spectra(all_results, PLOTS_DIR / '02_singular_spectra.png')
    plot_bottleneck_location(all_results, PLOTS_DIR / '03_bottleneck_location.png')
    plot_scaling_law(all_results, PLOTS_DIR / '04_scaling_law.png')
    plot_phase_diagram(all_results, PLOTS_DIR / '05_phase_diagram.png')
    plot_training_dynamics(all_results, PLOTS_DIR / '06_training_dynamics.png')  # Fixed: NC1 + Rank dual-axis
    plot_wd_ablation(all_results, PLOTS_DIR / '07_wd_ablation.png')
    plot_depth_analysis(all_results, PLOTS_DIR / '08_depth_analysis.png')
    plot_capacity_analysis(all_results, PLOTS_DIR / '09_capacity_analysis.png')
    plot_accuracy_vs_rank(all_results, PLOTS_DIR / '10_accuracy_vs_rank.png')
    plot_architecture_comparison(all_results, PLOTS_DIR / '11_architecture_comparison.png')
    plot_test_acc_vs_rank(all_results, PLOTS_DIR / '12_test_acc_vs_rank.png')
    plot_transfer_comparison(all_results, PLOTS_DIR / '13_transfer_comparison.png')  # Fixed: Pattern comparison
    
    # === NEW TRAINING PLOTS ===
    plot_training_curves_grid(all_results, PLOTS_DIR / '14_training_curves.png')
    plot_loss_curves_grid(all_results, PLOTS_DIR / '15_loss_curves.png')
    plot_generalization_gap(all_results, PLOTS_DIR / '16_generalization_gap.png')
    plot_learning_rate_schedule(all_results, PLOTS_DIR / '17_lr_schedule.png')
    plot_gradient_norms(all_results, PLOTS_DIR / '18_gradient_norms.png')
    plot_convergence_speed(all_results, PLOTS_DIR / '19_convergence_speed.png')
    
    # === EXPORT DATA ===
    export_comprehensive_results(all_results, RESULTS_DIR)
    
    # Save raw results
    with open(RESULTS_DIR / 'all_results.pkl', 'wb') as f:
        pickle.dump(all_results, f)
    
    print(f"\n✓ All {19} plots generated successfully!")
    print(f"✓ Results exported to {RESULTS_DIR}")
    
else:
    print("  No results to visualize")

#Only generate plots if sweep completed normally (not from fallback)
total_expected = np.prod([len(v) for v in sweep_params.values()]) * n_seeds

if len(all_results) >= total_expected:
    print("\n📊 Generating final comprehensive plots...")
    generate_all_plots(all_results, PLOTS_DIR, RESULTS_DIR)
else:
    print("\n⚠️  Partial results - plots already generated during fallback")
    print(f"    Current: {len(all_results)}/{total_expected} runs")

# %% Cell 18: Final Summary
print("\n" + "="*80)
print(" EXPERIMENT SUMMARY")
print("="*80)
if len(all_results) > 0:
    print(f"Total runs: {len(all_results)}")
    accs = [r['metrics'].get('test_acc_full', 0) for r in all_results]
    print(f"Mean Test Acc: {np.mean(accs):.3f}")
    print(f"Max Test Acc: {np.max(accs):.3f}")
print(" EXPERIMENT COMPLETE!")

✓ Imports successful! (CIFAR-10 Production Version)
✓ Device: cuda
✓ GPU: Tesla P100-PCIE-16GB
✓ GPU Memory: 17.06 GB
✓ Results directory: /kaggle/working/results
✓ Plots directory: /kaggle/working/plots

📦 INITIALIZING GLOBAL CIFAR-10 CACHE


100%|██████████| 170M/170M [00:10<00:00, 15.8MB/s]
Seed set to 42


✓ CIFAR-10 cached to cuda: Train=50000, Test=10000


 RUNNING QUICK TEST (should complete in <1 minute)
Testing with: n=20, d_b=16, L=3, max_iters=100


Seed set to 0


 TEST PASSED! Proceeding to full sweep...

 SMART SWEEP CONFIGURATION (Fits in 11.5h)
Total runs planned: 8400
Expected completion: ~9.5 - 11 hours

 Starting/Resuming sweep...

RESUMING SWEEP: 0/8400 runs already completed


[1/8400] n=5, d_b=8, L=3, wd=0, seed=0
⏱️  Elapsed: 0.00h | Remaining: 11.50h


Seed set to 1


... (2,431 outputs left out)

Seed set to 2



[1218/8400] n=10, d_b=16, L=3, wd=0, seed=2
⏱️  Elapsed: 10.67h | Remaining: 0.83h


Seed set to 0



[1219/8400] n=10, d_b=16, L=3, wd=1e-05, seed=0
⏱️  Elapsed: 10.72h | Remaining: 0.78h

 TIME WARNING: Only 0.74h remaining
 Saving checkpoint and generating plots...

💾 Checkpoint saved: 1219 results

 GENERATING VISUALIZATIONS FROM 1219 RUNS

✓ Saved: 01_rank_evolution.png
✓ Saved: 02_singular_spectra.png
✓ Saved: 03_bottleneck_location.png
✓ Saved: 04_scaling_law.png
✓ Saved: 05_phase_diagram.png
✓ Saved: 06_training_dynamics.png
✓ Saved: 07_wd_ablation.png
✓ Saved: 08_depth_analysis.png
✓ Saved: 09_capacity_analysis.png
✓ Saved: 10_accuracy_vs_rank.png
✓ Saved: 11_architecture_comparison.png
✓ Saved: 12_test_acc_vs_rank.png
✓ Saved: 13_transfer_comparison.png
✓ Saved: 14_training_curves.png
✓ Saved: 15_loss_curves.png
✓ Saved: 16_generalization_gap.png
✓ Saved: 17_lr_schedule.png
✓ Saved: 18_gradient_norms.png
✓ Saved: 19_convergence_speed.png
  Plot generation partially failed: name 'export_comprehensive_results' is not defined
 Results still saved to checkpoint

 GRACEFUL SHUTDO